<a href="https://colab.research.google.com/github/Tatinn4/CARFM/blob/main/Conditions_Builder(Datasets).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DocTamper Extraction: Condition 1 / 2 / 3 Dataset Builder
- **Condition 1 (Pristine Baseline):** raw images extracted as-is.
- **Condition 2 (Re-encoded):** pristine images re-compressed at `QUALITY_FACTOR(QF)`.
- **Condition 3 (PDF Round-trip):** PDF and extracted back out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install lmdb pymupdf --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.8/346.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 68.9 MB/s eta 0:00:00


In [ ]:
import os
import lmdb
import cv2
import pymupdf as fitz
import numpy as np
from io import BytesIO
from PIL import Image

In [ ]:
LMDB_PATH = '/content/drive/MyDrive/ADCD-Net/DocTamperV1/DocTamperV1-TestingSet'
OUTPUT_ROOT = '/content/drive/MyDrive/ADCD-Net/DocTamperConditions'

START_INDEX = 1
MAX_IMAGES = 10      # set to None to extract every remaining sample
QUALITY_FACTOR = 70

##Condition 1 (Pristine Baseline): extract raw images + masks from the LMDB

In [ ]:
def extract_condition1(lmdb_path, output_root, start_index=0, max_images=None):
    """Extract raw image/mask pairs from the DocTamper LMDB, unmodified."""
    images_dir = os.path.join(output_root, 'condition1_pristine', 'images')
    masks_dir = os.path.join(output_root, 'masks')
    os.makedirs(images_dir, exist_ok=True)
    os.makedirs(masks_dir, exist_ok=True)

    env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False, meminit=False)
    try:
        with env.begin(write=False) as txn:
            n_samples = int(txn.get(b'num-samples'))
        end_index = n_samples if max_images is None else min(start_index + max_images, n_samples)
        print(f'Extracting indices {start_index} to {end_index - 1} of {n_samples} samples')

        with env.begin(write=False) as txn:
            for index in range(start_index, end_index):
                img_bytes = txn.get(f'image-{index:09d}'.encode())
                mask_bytes = txn.get(f'label-{index:09d}'.encode())
                if img_bytes is None or mask_bytes is None:
                    print(f'Skipping index {index}: missing data')
                    continue

                with open(os.path.join(images_dir, f'{index:09d}.jpg'), 'wb') as f:
                    f.write(img_bytes)

                mask = cv2.imdecode(np.frombuffer(mask_bytes, np.uint8), 0)
                if mask.max() == 1:
                    mask = mask * 255
                cv2.imwrite(os.path.join(masks_dir, f'{index:09d}.png'), mask)
    finally:
        env.close()

    print(f'Condition 1 images: {images_dir}')
    print(f'Masks (shared by all conditions): {masks_dir}')
    return images_dir, masks_dir

##Condition 2 (Re-encoded) and Condition 3 (PDF Round-trip)

In [ ]:
def reencode_condition2(image_bytes, quality_factor):
    """Decode pristine JPEG bytes and re-encode at `quality_factor`. No PDF involved."""
    image = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)
    ok, encoded = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, quality_factor])
    if not ok:
        raise RuntimeError('JPEG encoding failed')
    return encoded.tobytes()


def pdf_roundtrip_condition3(jpeg_bytes):
    """
    Embed pre-compressed JPEG bytes into a one-page PDF and extract them back out.
    The bytes are passed via `stream=`, so PyMuPDF embeds them as-is instead of
    re-encoding -- this keeps Condition 3 at the same Quality Factor as Condition 2.
    """
    with Image.open(BytesIO(jpeg_bytes)) as im:
        width, height = im.size

    doc = fitz.open()
    page = doc.new_page(width=width, height=height)
    page.insert_image(fitz.Rect(0, 0, width, height), stream=jpeg_bytes)
    pdf_bytes = doc.tobytes()
    doc.close()

    doc2 = fitz.open(stream=pdf_bytes, filetype='pdf')
    xref = doc2.load_page(0).get_images(full=True)[0][0]
    extracted_bytes = doc2.extract_image(xref)['image']
    doc2.close()
    return extracted_bytes

##Quality Factor check

JPEG does not store its Quality Factor as a literal number it is recovered from the embedded quantization table. Used to confirm Condition 2 and Condition 3 really share the same Quality Factor.

In [ ]:
_BASE_LUMINANCE_TABLE = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
], dtype=float)


def estimate_quality_factor(jpeg_bytes):
    """Estimate the embedded JPEG Quality Factor from its luminance quantization table."""
    with Image.open(BytesIO(jpeg_bytes)) as im:
        table = np.array(im.quantization[0], dtype=float)
    scale = np.median(table / _BASE_LUMINANCE_TABLE) * 100
    qf = (200 - scale) / 2 if scale <= 100 else 5000 / scale
    return int(np.clip(round(qf), 1, 100))

In [ ]:
def build_conditions_2_and_3(condition1_images_dir, output_root, quality_factor):
    cond2_dir = os.path.join(output_root, 'Reencode', 'images')
    cond3_dir = os.path.join(output_root, 'Pdf-roundtrip', 'images')
    os.makedirs(cond2_dir, exist_ok=True)
    os.makedirs(cond3_dir, exist_ok=True)

    filenames = sorted(os.listdir(condition1_images_dir))
    qf_mismatches = []

    for filename in filenames:
        with open(os.path.join(condition1_images_dir, filename), 'rb') as f:
            pristine_bytes = f.read()

        cond2_bytes = reencode_condition2(pristine_bytes, quality_factor)
        cond3_bytes = pdf_roundtrip_condition3(cond2_bytes)

        with open(os.path.join(cond2_dir, filename), 'wb') as f:
            f.write(cond2_bytes)
        with open(os.path.join(cond3_dir, filename), 'wb') as f:
            f.write(cond3_bytes)

        cond2_qf = estimate_quality_factor(cond2_bytes)
        cond3_qf = estimate_quality_factor(cond3_bytes)
        if abs(cond2_qf - cond3_qf) > 1:
            qf_mismatches.append((filename, cond2_qf, cond3_qf))

    print(f'Condition 2 images: {cond2_dir}')
    print(f'Condition 3 images: {cond3_dir}')
    if qf_mismatches:
        print(f'WARNING: QF mismatch on {len(qf_mismatches)} of {len(filenames)} images:')
        for name, q2, q3 in qf_mismatches[:5]:
            print(f'  {name}: Condition2 QF={q2}, Condition3 QF={q3}')
    else:
        print(f'QF check passed: all {len(filenames)} images match at Q={quality_factor}.')

    return cond2_dir, cond3_dir

## 9. Run the pipeline

In [ ]:
condition1_images_dir = '/content/drive/MyDrive/ADCD-Net/DocTamperRaw/images'

condition2_images_dir, condition3_images_dir = build_conditions_2_and_3(
    condition1_images_dir, '/content/drive/MyDrive/ADCD-Net/DocTamperRaw', QUALITY_FACTOR
)

In [ ]:
import hashlib

def compute_psnr(image_a, image_b):
    mse = np.mean((image_a.astype(float) - image_b.astype(float)) ** 2)
    return float('inf') if mse == 0 else 20 * np.log10(255.0 / np.sqrt(mse))


def verify_conditions(condition1_images_dir, condition2_images_dir, condition3_images_dir, sample_size=10):
    """Compare a sample of images across condition pairs: byte-identity + PSNR."""
    filenames = sorted(os.listdir(condition1_images_dir))[:sample_size]
    pairs = [
        ('Condition 1 vs 2', condition1_images_dir, condition2_images_dir),
        ('Condition 1 vs 3', condition1_images_dir, condition3_images_dir),
        ('Condition 2 vs 3', condition2_images_dir, condition3_images_dir),
    ]

    for label, dir_a, dir_b in pairs:
        identical_count = 0
        psnr_values = []
        for filename in filenames:
            bytes_a = open(os.path.join(dir_a, filename), 'rb').read()
            bytes_b = open(os.path.join(dir_b, filename), 'rb').read()

            if hashlib.sha256(bytes_a).hexdigest() == hashlib.sha256(bytes_b).hexdigest():
                identical_count += 1

            img_a = cv2.imdecode(np.frombuffer(bytes_a, np.uint8), cv2.IMREAD_COLOR)
            img_b = cv2.imdecode(np.frombuffer(bytes_b, np.uint8), cv2.IMREAD_COLOR)
            psnr_values.append(compute_psnr(img_a, img_b))

        avg_psnr = np.mean([p for p in psnr_values if p != float('inf')] or [float('inf')])
        print(f'{label}: {identical_count}/{len(filenames)} byte-identical, avg PSNR={avg_psnr:.2f} dB')


verify_conditions(condition1_images_dir, condition2_images_dir, condition3_images_dir, sample_size=20)

Condition 1 vs 2: 0/10 byte-identical, avg PSNR=36.15 dB
Condition 1 vs 3: 0/10 byte-identical, avg PSNR=36.15 dB
Condition 2 vs 3: 10/10 byte-identical, avg PSNR=inf dB
